In [1]:
from __future__ import annotations

import numpy as np
import pandas as pd
from scipy import stats
from scipy.cluster import hierarchy
from scipy.spatial.distance import squareform
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import shap
import statsmodels.api as sm
import lightgbm as lgb

In [2]:
import warnings
warnings.filterwarnings('ignore')

In [3]:
pd.set_option("display.width", 220); pd.set_option("display.max_columns", None)
fmt = "{:.4f}".format

In [4]:
rng = np.random.default_rng(7)
n = 20_000

# ----- features ----------------------------------------------------
price = rng.gamma(4, 25, n)                                   # real effect
price_dup = price * 7.2 + rng.normal(0, 90, n)                # twin, NO own effect
rating = np.clip(rng.normal(4.0, 0.6, n), 1, 5)               # real effect
z_r = (rating - rating.mean()) / rating.std()
popularity = 50 + 15 * (0.75 * z_r + rng.normal(0, 0.6614, n))  # proxy, NO own effect
novelty = rng.uniform(0, 10, n)                               # inverted-U effect
delivery = rng.integers(1, 10, n).astype(float)               # effect ONLY on mobile
noise = rng.normal(0, 1, n)                                   # nothing
brand = rng.choice(["acme", "globex", "initech", "umbrella"], n, p=[.4, .3, .2, .1])
device = rng.choice(["mobile", "desktop", "tablet"], n, p=[.6, .3, .1])

# ----- true click mechanism (what the analysis should recover) -----
brand_eff = {"acme": 0.3, "globex": 0.0, "initech": -0.3, "umbrella": 0.6}
logit = (-2.1
            - 0.012 * (price - 100)                      # linear price effect
            + 0.9 * (rating - 4.0)                       # linear rating effect
            + 0.5 - 0.055 * (novelty - 5.0) ** 2         # inverted U in novelty
            - 0.16 * (delivery - 5.0) * (device == "mobile")   # pure interaction
            + pd.Series(brand).map(brand_eff).to_numpy())
click = rng.binomial(1, 1 / (1 + np.exp(-logit)))

In [5]:
df = pd.DataFrame({"f_price": price, "f_price_dup": price_dup,
                    "f_rating": rating, "f_popularity": popularity,
                    "f_novelty": novelty, "f_delivery_days": delivery,
                    "f_noise": noise, "brand": brand, "device": device,
                    "click": click})

numeric_cols = ["f_price", "f_price_dup", "f_rating", "f_popularity", "f_novelty", "f_delivery_days", "f_noise"]

categorical_cols = ["brand", "device"]

print(f"n={n}, CTR={df['click'].mean():.3f}")

n=20000, CTR=0.159


### 1. association matrix + heatmap

In [6]:
def correlation_ratio(cat, y) -> float:
    """eta = sqrt(between-group SS / total SS): association categorical->numeric.
    For a binary y this equals |point-biserial r|; it is the ANOVA R."""
    y = np.asarray(y, float)
    cat = np.asarray(cat)
    ybar = y.mean()
    sst = ((y - ybar) ** 2).sum()
    if sst == 0:
        return 0.0
    ssb = 0.0
    for g in np.unique(cat):
        yg = y[cat == g]
        ssb += len(yg) * (yg.mean() - ybar) ** 2
    return float(np.sqrt(ssb / sst))


def cramers_v_pair(a, b) -> float:
    """Cramer's V between two categorical series (no Yates correction:
    we want an unbiased *measure*, not a conservative test)."""
    ct = pd.crosstab(pd.Series(a).astype(str), pd.Series(b).astype(str))
    chi2 = stats.chi2_contingency(ct, correction=False)[0]
    n = ct.values.sum()
    denom = n * (min(ct.shape) - 1)
    return float(np.sqrt(chi2 / denom)) if denom > 0 else 0.0


def association_matrix(df: pd.DataFrame, numeric_cols, categorical_cols) -> pd.DataFrame:
    """Symmetric matrix in [0,1]. num-num: |Pearson|; num-cat: eta; cat-cat: V.
    All three reduce to the same 'shared variance' family, so mixing them on
    one scale is meaningful for *screening* redundancy (not exact equivalence)."""
    cols = list(numeric_cols) + list(categorical_cols)
    A = pd.DataFrame(np.eye(len(cols)), index=cols, columns=cols)
    for i, ci in enumerate(cols):
        for cj in cols[i + 1:]:
            ci_num, cj_num = ci in numeric_cols, cj in numeric_cols
            sub = df[[ci, cj]].dropna()
            if ci_num and cj_num:
                v = abs(stats.pearsonr(sub[ci], sub[cj])[0])
            elif ci_num != cj_num:                       # mixed pair
                num, cat = (ci, cj) if ci_num else (cj, ci)
                v = correlation_ratio(sub[cat], sub[num])
            else:
                v = cramers_v_pair(sub[ci], sub[cj])
            A.loc[ci, cj] = A.loc[cj, ci] = v
    return A

In [7]:
A = association_matrix(df, numeric_cols, categorical_cols)

print("\n=== [1] association matrix (|Pearson| / eta / Cramer's V) ===\n")
print(A.round(2))

fig, ax = plt.subplots(figsize=(7.5, 6.5))
im = ax.imshow(A.values, vmin=0, vmax=1, cmap="viridis")
ax.set_xticks(range(A.shape[1])); ax.set_xticklabels(A.columns, rotation=45, ha="right")
ax.set_yticks(range(A.shape[0])); ax.set_yticklabels(A.index)
for i in range(len(A)):
    for j in range(len(A)):
        ax.text(j, i, f"{A.values[i, j]:.2f}", ha="center", va="center",
                color="w" if A.values[i, j] < 0.6 else "k", fontsize=7)
fig.colorbar(im); fig.tight_layout(); plt.show()


=== [1] association matrix (|Pearson| / eta / Cramer's V) ===

                 f_price  f_price_dup  f_rating  f_popularity  f_novelty  f_delivery_days  f_noise  brand  device
f_price             1.00         0.97      0.01          0.01       0.00             0.00     0.01   0.01    0.01
f_price_dup         0.97         1.00      0.01          0.01       0.00             0.00     0.01   0.01    0.01
f_rating            0.01         0.01      1.00          0.75       0.01             0.01     0.01   0.02    0.01
f_popularity        0.01         0.01      0.75          1.00       0.00             0.01     0.00   0.02    0.01
f_novelty           0.00         0.00      0.01          0.00       1.00             0.00     0.02   0.01    0.01
f_delivery_days     0.00         0.00      0.01          0.01       0.00             1.00     0.00   0.00    0.01
f_noise             0.01         0.01      0.01          0.00       0.02             0.00     1.00   0.01    0.00
brand               0.01

### 2. clustering (before VIF so we can report cluster ids)

In [8]:
# ======================================================================
# Feature clustering on distance = 1 - association
# ======================================================================
def cluster_features(assoc: pd.DataFrame, threshold: float = 0.7):
    """Average-linkage hierarchical clustering; features with association
    > threshold end up in one cluster. Returns (labels Series, linkage Z)."""
    D = 1.0 - assoc.values
    D = (D + D.T) / 2.0 # make D symmetric (despite indeed it is)
    np.fill_diagonal(D, 0.0)
    Z = hierarchy.linkage(squareform(D, checks=False), method="average")
    labels = hierarchy.fcluster(Z, t=1.0 - threshold, criterion="distance")
    return pd.Series(labels, index=assoc.index, name="cluster"), Z

In [9]:
clusters, Z = cluster_features(A, threshold=0.7)

print("\n=== [3] feature clusters (association > 0.7 merged) ===\n")
for cid in sorted(clusters.unique()):
    print(f"  cluster {cid}: {list(clusters.index[clusters == cid])}")
fig, ax = plt.subplots(figsize=(8, 4))
hierarchy.dendrogram(Z, labels=list(A.index), ax=ax, color_threshold=0.3)
ax.axhline(0.3, ls="--", c="gray"); ax.set_ylabel("distance = 1 - association")
fig.tight_layout(); plt.show()


=== [3] feature clusters (association > 0.7 merged) ===

  cluster 1: ['f_novelty']
  cluster 2: ['f_noise']
  cluster 3: ['f_rating', 'f_popularity']
  cluster 4: ['brand']
  cluster 5: ['f_price', 'f_price_dup']
  cluster 6: ['device']
  cluster 7: ['f_delivery_days']


### 3. encoding shared by every model

In [10]:
def encode_features(df, numeric_cols, categorical_cols, standardize=True):
    """z-score numerics (comparable coefficients), one-hot categoricals
    (drop_first -> reference category, avoids the exact-collinearity trap).
    Returns (X, mapping original feature -> its encoded column list)."""
    parts, mapping = [], {}
    for c in numeric_cols:
        x = df[c].astype(float)
        parts.append(((x - x.mean()) / x.std(ddof=0) if standardize else x).rename(c))
        mapping[c] = [c]
    for c in categorical_cols:
        d = pd.get_dummies(df[c].astype(str), prefix=c, drop_first=True).astype(float)
        parts.append(d)
        mapping[c] = list(d.columns)
    X = pd.concat(parts, axis=1)
    return X, mapping

In [11]:
X, mapping = encode_features(df, numeric_cols, categorical_cols)
y = df["click"]

In [12]:
X.head()

,f_price,f_price_dup,f_rating,f_popularity,f_novelty,f_delivery_days,f_noise,brand_globex,brand_initech,brand_umbrella,device_mobile,device_tablet
0,-0.160671,0.021472,0.524042,1.067316,-0.338857,-0.001738,-1.103871,0.0,1.0,0.0,1.0,0.0
1,-0.413844,-0.302319,-0.288260,-0.064975,-1.670267,1.169938,0.820090,0.0,0.0,0.0,1.0,0.0
2,-0.566591,-0.676375,-0.001128,0.019538,1.361711,1.560496,-0.049727,0.0,0.0,0.0,1.0,0.0
3,-0.103241,-0.348847,-0.588986,-1.014736,-1.511129,1.169938,1.128747,0.0,0.0,0.0,0.0,1.0
4,-0.597072,-0.505009,-0.643050,0.507254,0.608122,1.169938,1.554355,0.0,0.0,0.0,1.0,0.0


### 4. VIF

1. **Build a standard multiple regression model** using all the other features ($X_2, X_3, \dots, X_n$) to predict $X_1$.

   1.1 **Calculate the Sum of Squares:**
   * Total Sum of Squares (SST):
     $$SST = \sum (X_1 - \bar{X}_1)^2$$
   * Residual Sum of Squares (SSR / SSE):
     $$SSR = \sum (X_1 - \hat{X}_1)^2$$
     *(where $\hat{X}_1$ is the model's prediction)*
   * **SSR** represents the **Unexplained Variance**—the amount of variation in $X_1$ that the other features ($X_2, X_3$, etc.) completely failed to predict.

   1.2 **Calculate $R^2$:**
   $$R^2 = 1 - \frac{SSR}{SST}$$
   * **If $R^2 = 0$:** The other features ($X_2, X_3$) offer absolutely no clues about the value of $X_1$.
     * **VIF:** $\frac{1}{1 - 0} = \mathbf{1}$. The feature brings 100% unique, independent information to your final model.
   * **If $R^2 = 0.99$:** 99% of the information contained in feature $X_1$ is already captured by looking at $X_2, X_3$, etc. $X_1$ is effectively a duplicate.
     * **VIF:** $\frac{1}{1 - 0.99} = \frac{1}{0.01} = \mathbf{100}$. This triggers a massive VIF score, warning you that keeping $X_1$ will artificially inflate the variance of your coefficients and make your final model unstable.

2. **Calculate VIF:**
   $$VIF_i = \frac{1}{1 - R_i^2}$$
   *(The Variance Inflation Factor for a specific feature $i$)*

3. **Standard Interpretation Guide:**
   * **VIF = 1:** Perfect. The feature is completely independent of all other features.
   * **1 < VIF < 5:** Moderate correlation. Generally acceptable, but worth noting.
   * **5 $\le$ VIF $\le$ 10:** High correlation. This is the danger zone; you should investigate these features.
   * **VIF > 10:** Severe multicollinearity. The regression coefficients are poorly estimated, and the feature is almost entirely redundant.

In [13]:
def compute_vif(X: pd.DataFrame) -> pd.Series:
    """VIF_j = 1 / (1 - R2_j), R2_j from regressing column j on all others.
    sqrt(VIF_j) = factor by which collinearity inflates SE(beta_j)."""
    out = {}
    for j, col in enumerate(X.columns):
        others = X.drop(columns=col)
        r2 = LinearRegression().fit(others, X[col]).score(others, X[col])
        out[col] = np.inf if r2 >= 1.0 else 1.0 / (1.0 - r2)
    return pd.Series(out, name="VIF").sort_values(ascending=False)

In [14]:
vif = compute_vif(X)
print(vif.round(2).to_string())

f_price_dup        16.82
f_price            16.81
f_popularity        2.32
f_rating            2.32
brand_globex        1.22
brand_initech       1.20
device_mobile       1.20
device_tablet       1.20
brand_umbrella      1.12
f_novelty           1.00
f_noise             1.00
f_delivery_days     1.00


### 5. logistic regression 

#### The Logistic Regression Model

The model estimates the log-odds ($\text{logit}$) of the target outcome $y = 1$:

$$\ln\left(\frac{P}{1 - P}\right) = \beta_0 + \beta_1 X_1 + \dots + \beta_j X_j + \dots + \beta_p X_p$$

#### Defining Odds

Odds are defined as the ratio of the probability of the event occurring to the probability of it not occurring:

$$\text{Odds} = \frac{P}{1 - P} = e^{\beta_0 + \beta_1 X_1 + \dots + \beta_j X_j + \dots + \beta_p X_p}$$

#### Calculating the Ratio for a 1-Unit Increase

When feature $X_j$ increases by 1 unit (or 1 standard deviation, since your numerical features are z-scored) from $x$ to $x + 1$:

$$\text{Odds Ratio}_j = \frac{\text{Odds}(X_j = x + 1)}{\text{Odds}(X_j = x)} = \frac{e^{\beta_0 + \dots + \beta_j(x + 1) + \dots}}{e^{\beta_0 + \dots + \beta_j(x) + \dots}} = e^{\beta_j}$$

* **`odds_ratio > 1` (Positive effect):** e.g., $1.62$ means the odds of conversion increase by $62\%$.
* **`odds_ratio < 1` (Negative effect):** e.g., $0.53$ means the odds of conversion decrease by $47\%$ (or are $53\%$ of the baseline).
* **`odds_ratio = 1` (No effect):** $0\%$ change on the odds.

### z-score

$$z = \frac{\text{coef}}{\text{SE}}$$

### $p$ ($p$-value)

* **What it is:** The probability of observing a coefficient at least as extreme as the one in your table if the variable actually had zero effect in reality.
* **How to read it:**
  * **$p < 0.05$ (Statistically Significant):** Reject the null hypothesis. The feature has a statistically meaningful relationship with the outcome.
  * **$p \ge 0.05$ (Not Significant):** Fail to reject the null hypothesis. The observed effect could just be random noise.

### `OR_lo95` & `OR_hi95` (95% Confidence Interval for the Odds Ratio)

### SE (Standard Error) - A measure of the statistical uncertainty of the estimated coefficient (coef).
### How to calculate SE:

According to asymptotic normality theory for Maximum Likelihood Estimators (MLEs), as the sample size $N \to \infty$, the maximum likelihood estimator $\hat{\boldsymbol{\beta}}$ converges in distribution to a normal distribution centered at the true parameter $\boldsymbol{\beta}$:

$$\hat{\boldsymbol{\beta}} \xrightarrow{d} \mathcal{N}\left(\boldsymbol{\beta}, \, I(\boldsymbol{\beta})^{-1}\right)$$

$$\text{Var}(\hat{\boldsymbol{\beta}}) = I(\boldsymbol{\beta})^{-1}$$

$$I(\boldsymbol{\beta}) = X^T W X$$

where:

$$W = \begin{bmatrix} p_1(1-p_1) & 0 & \dots & 0 \\ 0 & p_2(1-p_2) & \dots & 0 \\ \vdots & \vdots & \ddots & \vdots \\ 0 & 0 & \dots & p_N(1-p_N) \end{bmatrix}$$

Since in our case we have Bernoulli trials, the variance of each Bernoulli trial (a $0/1$ outcome) is given by $p_i(1 - p_i)$.

In [15]:
# ======================================================================
# Multivariate additive: logistic regression with inference
# ======================================================================
def fit_logistic_inference(X: pd.DataFrame, y: pd.Series):
    """statsmodels Logit -> tidy table with coef, SE, z, p, odds ratio, CI.
    Numerics are z-scored, so exp(coef) = odds multiplier per +1 SD,
    holding every other column fixed (that clause IS the adjustment)."""
    res = sm.Logit(y.astype(float), sm.add_constant(X)).fit(disp=0)
    tab = pd.DataFrame({
        "coef": res.params, "SE": res.bse, "z": res.tvalues, "p": res.pvalues,
        "odds_ratio": np.exp(res.params),
        "OR_lo95": np.exp(res.params - 1.96 * res.bse),
        "OR_hi95": np.exp(res.params + 1.96 * res.bse),
    })
    return res, tab.drop(index="const")

### The table, row by row, in plain English:

| Row | Reading | Verdict |
| :--- | :--- | :--- |
| **`f_price`** | $+1$ SD ($\approx +\$50$) multiplies odds by $0.53$ $[0.45, 0.62]$ — cuts them roughly in half | Real driver $(-)$ |
| **`f_price_dup`** | $\times 1.08$ $[0.92, 1.27]$, $p = 0.35$ — nothing beyond `f_price` | Redundant twin |
| **`f_rating`** | $+1$ SD ($\approx +0.6\star$) multiplies odds by $1.62$ $[1.52, 1.72]$ | Real driver $(+)$ |
| **`f_popularity`** | $\times 1.01$ $[0.95, 1.07]$ — unique contribution bounded within $\pm 7\%$ | Proxy of rating |
| **`f_novelty`** | Linear term nil, $p = 0.47$ — but SHAP ranks it #2 | U-shape; invisible to this table by design |
| **`f_delivery_days`** | $+1$ SD ($\approx +2.6$ days) $\to$ odds $\times 0.80$ $[0.77, 0.84]$ | Real; the traffic-weighted average of a mobile-only effect |
| **`f_noise`** | $\times 1.02$ $[0.98, 1.06]$ — bounded nothing | Truly nothing (all methods agree) |
| **`brand_globex`** | vs Acme: $\times 0.75$ $[0.68, 0.83]$ | Worse than reference |
| **`brand_initech`** | vs Acme: $\times 0.58$ $[0.52, 0.66]$ | Worst brand |
| **`brand_umbrella`** | vs Acme: $\times 1.33$ $[1.17, 1.51]$ | Best brand |
| **`device_mobile` / `tablet`** | $\approx 1$, null | Main effect $\approx 0$ — device matters only through the delivery interaction ($\text{LRT } p \approx 10^{-18}$) |

In [16]:
print("\n=== [4] logistic regression, z-scored numerics (adjusted effects) ===\n")
logit_res, logit_tab = fit_logistic_inference(X, y)
with pd.option_context("display.float_format", fmt):
    print(logit_tab)


=== [4] logistic regression, z-scored numerics (adjusted effects) ===

                   coef     SE        z      p  odds_ratio  OR_lo95  OR_hi95
f_price         -0.6345 0.0836  -7.5925 0.0000      0.5302   0.4501   0.6246
f_price_dup      0.0773 0.0823   0.9392 0.3476      1.0804   0.9194   1.2695
f_rating         0.4828 0.0316  15.2715 0.0000      1.6205   1.5232   1.7241
f_popularity     0.0101 0.0307   0.3286 0.7425      1.0101   0.9512   1.0728
f_novelty        0.0145 0.0201   0.7227 0.4698      1.0146   0.9755   1.0553
f_delivery_days -0.2197 0.0202 -10.8514 0.0000      0.8027   0.7715   0.8352
f_noise          0.0176 0.0201   0.8773 0.3803      1.0178   0.9785   1.0587
brand_globex    -0.2866 0.0489  -5.8605 0.0000      0.7508   0.6822   0.8263
brand_initech   -0.5364 0.0587  -9.1335 0.0000      0.5848   0.5212   0.6562
brand_umbrella   0.2876 0.0646   4.4514 0.0000      1.3332   1.1746   1.5132
device_mobile    0.0183 0.0448   0.4078 0.6834      1.0184   0.9328   1.1119
devi

### 6. GBM + SHAP

#### A. The Shapley Value Formula

The SHAP value $\phi_i$ assigned to feature $i$ for a specific observation $x$ is defined as:

$$\phi_i(x) = \sum_{S \subseteq F \setminus \{i\}} \frac{|S|!(|F| - |S| - 1)!}{|F|!} \Big[ f_x(S \cup \{i\}) - f_x(S) \Big]$$

Where:
* **$F$**: The set of all $P$ features.
* **$S$**: A subset of features that does not include feature $i$.
* **$f_x(S)$**: The model's expected prediction when only the subset of features $S$ is present/known.
* **$f_x(S \cup \{i\}) - f_x(S)$**: The marginal contribution of feature $i$ when added to subset $S$.
* **$\frac{|S|!(|F| - |S| - 1)!}{|F|!}$**: A weighting factor representing the probability that subset $S$ occurs when ordering features randomly.

#### B. The Additive Feature Attribution Property

SHAP values satisfy the efficiency property, meaning the sum of all feature SHAP values equals the difference between the model's prediction $f(x)$ and the baseline expected prediction $\mathbb{E}[f(X)]$:

$$f(x) = \mathbb{E}[f(X)] + \sum_{i=1}^{P} \phi_i(x)$$

Where:
* **$\mathbb{E}[f(X)]$ (Base Value):** The average prediction of the model across the training set (in log-odds for classification).
* **$\phi_i(x)$:** The contribution of feature $i$ pushing the prediction up or down from the base value.

$$\mathbb{E}[f(X)] = \frac{1}{N} \sum_{k=1}^{N} f(x^{(k)})$$

#### How to Read an Individual SHAP Value ($\phi_i$)

* **Positive SHAP Value ($\phi_i > 0$):**
  The feature value for this specific row pushed the prediction higher than the baseline average (i.e., increased the log-odds and probability of $y = 1$).

* **Negative SHAP Value ($\phi_i < 0$):**
  The feature value for this specific row pushed the prediction lower than the baseline average (i.e., decreased the log-odds and probability of $y = 1$).

* **Zero SHAP Value ($\phi_i = 0$):**
  This feature had no impact on shifting the prediction away from the baseline average for this row.

In [ ]:
def fit_gbm(X_tr, y_tr, X_te, y_te, seed=0):
    model = lgb.LGBMClassifier(
        n_estimators=400, learning_rate=0.05, num_leaves=31,
        class_weight="balanced", random_state=seed, verbose=-1)
    model.fit(X_tr, y_tr)
    return model, roc_auc_score(y_te, model.predict_proba(X_te)[:, 1])


def shap_values_of(model, X):
    """Return (n, p) SHAP matrix in log-odds units, robust to SHAP versions
    that return a list [class0, class1] instead of one array."""
    sv = shap.TreeExplainer(model).shap_values(X)
    if isinstance(sv, list):
        sv = sv[1]
    if sv.ndim == 3:                      # (n, p, 2) layout in some versions
        sv = sv[:, :, 1]
    return sv


def shap_interaction_ranking(model, X, top=6):
    """Mean |SHAP interaction value| per pair (i<j) -> top interacting pairs.
    Exact for trees; O(n * p^2), so pass a subsample."""
    iv = shap.TreeExplainer(model).shap_interaction_values(X)
    if isinstance(iv, list):
        iv = iv[1]
    M = np.abs(iv).mean(axis=0)           # (p, p)
    rows = []
    cols = list(X.columns)
    for i in range(len(cols)):
        for j in range(i + 1, len(cols)):
            rows.append((cols[i], cols[j], M[i, j]))
    return (pd.DataFrame(rows, columns=["feat_a", "feat_b", "mean_abs_interaction"])
            .sort_values("mean_abs_interaction", ascending=False).head(top)
            .reset_index(drop=True))

In [25]:
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=0)
model, auc_te = fit_gbm(X_tr, y_tr, X_te, y_te)
print(f"\n=== [5] LightGBM test AUC = {auc_te:.4f} ===\n")
sv = shap_values_of(model, X_te)

shap_mean = pd.Series(np.abs(sv).mean(axis=0), index=X_te.columns, name="mean_abs_shap").sort_values(ascending=False)
print(shap_mean.round(4).to_string())

shap.summary_plot(sv, X_te, show=False, max_display=12)
plt.tight_layout(); plt.savefig("shap_beeswarm.png", dpi=120); plt.close()

# scatter plot that shows how a single feature (f_novelty) affects the model's predictions across dataset
shap.dependence_plot("f_novelty", sv, X_te, interaction_index=None, show=False)
plt.tight_layout(); plt.savefig("shap_dependence_novelty.png", dpi=120); plt.close()

# make sure real U shape plot
shap.dependence_plot("f_novelty", sv, X_te, interaction_index="auto", show=False)
plt.tight_layout(); plt.savefig("f_novelty_u_shap.png", dpi=120); plt.close()

# scatter plot showing the effect of delivery days (f_delivery_days) on model predictions, 
# while color-coding each point by a second feature (device_mobile) to reveal feature interactions.
shap.dependence_plot("f_delivery_days", sv, X_te, interaction_index="device_mobile", show=False)
plt.tight_layout(); plt.savefig("shap_dependence_delivery.png", dpi=120); plt.show()


=== [5] LightGBM test AUC = 0.7037 ===

f_price            0.5047
f_novelty          0.4606
f_rating           0.4486
f_delivery_days    0.2060
f_price_dup        0.1836
brand_initech      0.1719
f_popularity       0.1321
f_noise            0.1237
brand_globex       0.1163
device_mobile      0.0674
device_tablet      0.0434
brand_umbrella     0.0286


### How to Interpret SHAP interaction pairs

The function outputs a table structured like this:

| `feat_a` | `feat_b` | `mean_abs_interaction` |
| :--- | :--- | :--- |
| `f_delivery_days` | `device_mobile` | 0.1452 |
| `f_price` | `f_rating` | 0.0981 |
| `brand_umbrella` | `f_price` | 0.0412 |

* **`feat_a` & `feat_b`:** The two features working together in the model.
* **`mean_abs_interaction`:** The average magnitude (in log-odds/margin space) by which the interaction term modifies the prediction. Higher values signify a stronger joint effect.

In [24]:
print("\n--- top SHAP interaction pairs (1200-row subsample) ---\n")
inter = shap_interaction_ranking(model, X_te.iloc[:1200])
with pd.option_context("display.float_format", fmt):
    print(inter)


--- top SHAP interaction pairs (1200-row subsample) ---

            feat_a         feat_b  mean_abs_interaction
0          f_price    f_price_dup                0.1125
1         f_rating   f_popularity                0.0663
2  f_delivery_days  device_mobile                0.0628
3         f_rating      f_novelty                0.0544
4          f_price      f_novelty                0.0523
5          f_price       f_rating                0.0519
